# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BarnaliBhowmik/FlyRank_ML/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### My response:

The action queue prioritizes pages for human review using the Week-4 transparent
baseline score.

The score combines the observed CTR gap for the page's position bucket with
log-scaled impressions. This gives higher priority to pages that appear to
under-capture clicks relative to similar-position pages while giving more
weight to pages with meaningful search exposure.

Each recommendation includes a reason code so that a reviewer can understand
why the page was ranked.

### Reason codes

- **LOW_CTR_FOR_POSITION** — observed CTR is below the benchmark for the
  page's position bucket.
- **NO_CTR_GAP** — no positive CTR gap was measured against the position
  benchmark.

### Suggested actions

- **REVIEW_CTR** — review the page's search-result presentation, title,
  description, and search intent before deciding whether a change is needed.
- **NO_ACTION** — no CTR-gap signal was measured by this baseline; the page
  does not receive an automatic content-change recommendation.

The queue is a prioritization tool for human review. A high score does not
prove that changing the page will increase CTR.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [3]:
!pip -q install duckdb huggingface_hub pyarrow

import duckdb
import pandas as pd
import numpy as np
import pyarrow.dataset as ds
from huggingface_hub import HfFileSystem
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

fs = HfFileSystem(token=HF_TOKEN)

print("Libraries loaded and Hugging Face access ready.")

Libraries loaded and Hugging Face access ready.


In [4]:
march_path = (
    "datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/"
)

march_files = fs.glob(march_path + "*.parquet")

print("March files found:", len(march_files))

march_file = march_files[0]
march_uri = "hf://" + march_file

march_dataset = ds.dataset(
    march_uri,
    format="parquet",
    filesystem=fs
)

con = duckdb.connect()
con.register("march_data", march_dataset)

print("March 2026 data connected successfully.")

March files found: 1
March 2026 data connected successfully.


In [5]:
page_data = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_pageviews) AS ga4_pageviews,
    SUM(ga4_sessions) AS ga4_sessions,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
    SUM(scroll_events) AS scroll_events,
    SUM(gsc_clicks) AS gsc_clicks,
    SUM(gsc_impressions) AS gsc_impressions
FROM march_data
WHERE gsc_data_available IS TRUE
  AND gsc_avg_position > 0
  AND gsc_avg_position <= 20
  AND gsc_impressions > 0
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

page_data["observed_ctr"] = (
    page_data["gsc_clicks"] /
    page_data["gsc_impressions"]
)

def position_bucket(position):
    if position <= 3:
        return "01-03"
    elif position <= 5:
        return "04-05"
    elif position <= 10:
        return "06-10"
    else:
        return "11-20"

page_data["position_bucket"] = (
    page_data["gsc_avg_position"].apply(position_bucket)
)

bucket_benchmarks = (
    page_data
    .groupby("position_bucket")
    .apply(
        lambda x:
        x["gsc_clicks"].sum() /
        x["gsc_impressions"].sum()
    )
    .to_dict()
)

page_data["benchmark_ctr"] = (
    page_data["position_bucket"]
    .map(bucket_benchmarks)
)

page_data["ctr_gap"] = (
    page_data["benchmark_ctr"] -
    page_data["observed_ctr"]
).clip(lower=0)

print("Page-level dataset created.")
print("Rows:", len(page_data))
print("Clients:", page_data["client_hash_id"].nunique())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Page-level dataset created.
Rows: 163911
Clients: 47


/tmp/ipykernel_1225/2237789554.py:44: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [6]:
# Build the ranked action queue from the Week-4 baseline logic.

action_queue = page_data.copy()

action_queue["baseline_score"] = (
    action_queue["ctr_gap"] *
    np.log1p(action_queue["gsc_impressions"])
)

action_queue["reason_code"] = np.where(
    action_queue["ctr_gap"] > 0,
    "LOW_CTR_FOR_POSITION",
    "NO_CTR_GAP"
)

action_queue["suggested_action"] = np.where(
    action_queue["ctr_gap"] > 0,
    "REVIEW_CTR",
    "NO_ACTION"
)

action_queue = (
    action_queue
    .sort_values(
        ["baseline_score", "gsc_impressions"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

action_queue["priority_rank"] = (
    np.arange(len(action_queue)) + 1
)

queue_columns = [
    "priority_rank",
    "client_hash_id",
    "content_hash_id",
    "gsc_avg_position",
    "gsc_impressions",
    "observed_ctr",
    "benchmark_ctr",
    "ctr_gap",
    "baseline_score",
    "reason_code",
    "suggested_action"
]

action_queue = action_queue[queue_columns]

print("Ranked action queue created.")
print("Rows:", len(action_queue))

display(action_queue.head(20))

Ranked action queue created.
Rows: 163911


,priority_rank,client_hash_id,content_hash_id,gsc_avg_position,gsc_impressions,observed_ctr,benchmark_ctr,ctr_gap,baseline_score,reason_code,suggested_action
0,1,client_1a730cb2640a1abf,content_d61fc394d10cba41,2.740744,38000.0,0.000026,0.004045,0.004019,0.042379,LOW_CTR_FOR_POSITION,REVIEW_CTR
1,2,client_73cda7b4e4f265ea,content_8e1334d6356668e3,4.545582,134984.0,0.000007,0.003499,0.003492,0.041249,LOW_CTR_FOR_POSITION,REVIEW_CTR
2,3,client_62f4a7e64f5e0096,content_fc67675904376267,2.261303,60172.0,0.000299,0.004045,0.003746,0.041223,LOW_CTR_FOR_POSITION,REVIEW_CTR
3,4,client_e547b89c05043229,content_306bc78dff1eb683,1.488604,80821.0,0.000433,0.004045,0.003612,0.040815,LOW_CTR_FOR_POSITION,REVIEW_CTR
4,5,client_fef1a8f436438636,content_66bf45eb0c5bb550,2.784944,24259.0,0.000041,0.004045,0.004004,0.040425,LOW_CTR_FOR_POSITION,REVIEW_CTR
5,6,client_73cda7b4e4f265ea,content_b9acd1ebff7d34ff,2.418270,25941.0,0.000116,0.004045,0.003929,0.039937,LOW_CTR_FOR_POSITION,REVIEW_CTR
6,7,client_73cda7b4e4f265ea,content_1d7764b642f7bb9f,1.843455,23402.0,0.000171,0.004045,0.003874,0.038976,LOW_CTR_FOR_POSITION,REVIEW_CTR
7,8,client_e547b89c05043229,content_c46df0fa61530d86,1.556258,70398.0,0.000597,0.004045,0.003448,0.038491,LOW_CTR_FOR_POSITION,REVIEW_CTR
8,9,client_a80fca3f171ed1de,content_fa17add7836d36c3,1.902457,12588.0,0.000000,0.004045,0.004045,0.038187,LOW_CTR_FOR_POSITION,REVIEW_CTR
9,10,client_62f4a7e64f5e0096,content_34a70fea29d15f24,3.219473,143019.0,0.000301,0.003499,0.003199,0.037970,LOW_CTR_FOR_POSITION,REVIEW_CTR


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### My response: Intended use

This playbook is intended for content or SEO reviewers who need to prioritize
which pages to inspect first.

The ranked queue is a decision-support tool. It highlights pages with a
measured CTR gap relative to other pages in the same position bucket and
combines that gap with search exposure to prioritize review.

A reviewer can use the queue to decide where to investigate:
- search-result presentation and metadata;
- whether the page matches the search intent;
- whether the page needs content improvement;
- whether the observed signal is meaningful enough to justify further review.

### Limits

The queue is based on March 2026 development data and the transparent
position-bucket baseline used in this project.

A high score does **not** mean that:
- the page is definitely underperforming;
- changing the page will increase CTR;
- the recommended action will produce business value;
- the result will generalize to future months or unseen clients.

The analysis is observational rather than experimental. It measures a
prioritization signal; it does not establish causality.

The current opportunity proxy is also derived from the same CTR-gap concept
used by the baseline, so the measured baseline performance should not be
interpreted as proof that the baseline is generally superior.

The playbook is therefore intended for human-reviewed prioritization, not
automatic content changes or production decision-making.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [8]:
# Check that the action queue contains the fields needed for human review.

required_columns = [
    "priority_rank",
    "gsc_avg_position",
    "gsc_impressions",
    "ctr_gap",
    "baseline_score",
    "reason_code",
    "suggested_action"
]

missing_columns = [
    column
    for column in required_columns
    if column not in action_queue.columns
]

print("Required queue columns:", len(required_columns))
print("Missing columns:", missing_columns)

if missing_columns:
    raise ValueError(
        f"Action queue is missing required columns: {missing_columns}"
    )

print("Intended-use queue check passed.")
print("The queue contains the fields needed for human-reviewed prioritization.")

Required queue columns: 7
Missing columns: []
Intended-use queue check passed.
The queue contains the fields needed for human-reviewed prioritization.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### My response: Human review rules

Every high-priority page should be reviewed by a human before any action is
taken.

The reviewer should check:

1. **Search intent** — Does the page actually match the intent suggested by
   the search traffic?
2. **Position context** — Is the page being compared with pages in a
   reasonable position range?
3. **CTR context** — Is the observed CTR gap large enough to be meaningful,
   rather than a small difference that may be noise?
4. **Search exposure** — Does the page have enough impressions for the signal
   to justify attention?
5. **Page quality** — Is there an obvious content, metadata, or presentation
   issue that a reviewer can verify?
6. **Business context** — Is changing the page appropriate for its role and
   importance?
7. **Recent changes** — Has the page recently been changed, published, or
   affected by another known event?

The queue should support the reviewer's decision, not replace it.

### No-go list: what should NOT be automated

The system should **not** automatically:

- rewrite page titles or descriptions;
- rewrite or delete content;
- publish changes to a page;
- decide that a page is low quality;
- delete, merge, or redirect a page;
- claim that a change will increase CTR;
- prioritize pages for business action without human review;
- treat a high score as proof of an SEO problem.

These decisions require context that is not represented by the current
scoring features.

The safe automation boundary is therefore **ranking and explanation only**:
the system can produce a review queue and reason codes, while a human decides
whether any action is appropriate.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [10]:
# Check that the queue provides both a reason and a suggested action
# for every ranked page.

review_check = action_queue[
    ["priority_rank", "reason_code", "suggested_action"]
].copy()

missing_reason = review_check["reason_code"].isna().sum()
missing_action = review_check["suggested_action"].isna().sum()

print("Rows in review queue:", len(review_check))
print("Missing reason codes:", missing_reason)
print("Missing suggested actions:", missing_action)

if missing_reason > 0 or missing_action > 0:
    raise ValueError(
        "Some queue rows do not have a reason code or suggested action."
    )

print("Human-review fields check passed.")
print("Every ranked row has a reason code and suggested action.")

Rows in review queue: 163911
Missing reason codes: 0
Missing suggested actions: 0
Human-review fields check passed.
Every ranked row has a reason code and suggested action.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### My response:

The playbook should be checked periodically because the underlying search and
engagement patterns can change.

### Monitoring checks

The following signals would indicate that the recommendations may be becoming
stale:

- **CTR distribution shift** — the observed CTR distribution changes
  substantially from the development period.
- **Position distribution shift** — the mix of pages across position buckets
  changes substantially.
- **Exposure shift** — the distribution of impressions changes, especially if
  many pages move into very low-volume ranges.
- **Queue composition shift** — the proportion of pages receiving each reason
  code changes substantially.
- **Performance drift** — Precision@20 on a newly evaluated period falls
  materially below the measured development result when a comparable label is
  available.

### Retrain / refresh triggers

The queue should be reconsidered or recalculated when:

1. A new evaluation period becomes available.
2. The CTR or position distributions show meaningful drift.
3. The ranking begins producing a substantially different action mix.
4. The underlying measurement or tracking definitions change.
5. A new leakage-safe target becomes available.

For this prototype, these are review triggers rather than automatic production
retraining rules. The current system is intended for periodic analysis and
human-reviewed decision support.

The queue should be regenerated for a new analysis period rather than assuming
that March 2026 rankings remain valid indefinitely.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [12]:
# Monitoring summary for the current action queue.
# These values provide a reference point for future periods.

monitoring_summary = pd.DataFrame({
    "metric": [
        "Rows in queue",
        "Median CTR",
        "Median impressions",
        "Median CTR gap",
        "Median baseline score",
        "Share LOW_CTR_FOR_POSITION"
    ],
    "March 2026 value": [
        len(action_queue),
        action_queue["observed_ctr"].median(),
        action_queue["gsc_impressions"].median(),
        action_queue["ctr_gap"].median(),
        action_queue["baseline_score"].median(),
        (
            action_queue["reason_code"]
            .eq("LOW_CTR_FOR_POSITION")
            .mean()
        )
    ]
})

display(monitoring_summary)

,metric,March 2026 value
0,Rows in queue,163911.000000
1,Median CTR,0.000000
2,Median impressions,108.000000
3,Median CTR gap,0.002927
4,Median baseline score,0.007507
5,Share LOW_CTR_FOR_POSITION,0.804260


In [13]:
# Save the current reference values for comparison with future periods.

monitoring_baseline = {
    "period": "2026-03",
    "rows": int(len(action_queue)),
    "median_ctr": float(action_queue["observed_ctr"].median()),
    "median_impressions": float(action_queue["gsc_impressions"].median()),
    "median_ctr_gap": float(action_queue["ctr_gap"].median()),
    "median_baseline_score": float(action_queue["baseline_score"].median()),
    "share_low_ctr_for_position": float(
        action_queue["reason_code"]
        .eq("LOW_CTR_FOR_POSITION")
        .mean()
    )
}

print("Monitoring reference created:")
for key, value in monitoring_baseline.items():
    print(f"{key}: {value}")

Monitoring reference created:
period: 2026-03
rows: 163911
median_ctr: 0.0
median_impressions: 108.0
median_ctr_gap: 0.002926888731928216
median_baseline_score: 0.007507321372320689
share_low_ctr_for_position: 0.8042596287009414


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### My response:

The ranked action queue is exported so that the paper can reuse the same
recommendations produced by this notebook.

The CSV contains the priority rank, measured CTR context, baseline score,
reason code, and suggested action for each page in the March 2026 development
queue.

The queue is regenerated by the notebook rather than committed to the
repository, because the repository's data-file guard keeps generated data
files out of Git.

The exported queue is a decision-support artifact, not a production action
list.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [15]:
from pathlib import Path

# Create the output directory if it does not already exist.
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Export the ranked action queue.
queue_path = output_dir / "w07_action_queue.csv"

action_queue.to_csv(
    queue_path,
    index=False
)

print("Action queue exported successfully.")
print("File:", queue_path)
print("Rows:", len(action_queue))
print("Columns:", len(action_queue.columns))

Action queue exported successfully.
File: work/outputs/w07_action_queue.csv
Rows: 163911
Columns: 11


In [16]:
import json

metrics_path = output_dir / "w07_monitoring_baseline.json"

with open(metrics_path, "w") as f:
    json.dump(
        monitoring_baseline,
        f,
        indent=2
    )

print("Monitoring baseline exported successfully.")
print("File:", metrics_path)

with open(metrics_path, "r") as f:
    print(f.read())

Monitoring baseline exported successfully.
File: work/outputs/w07_monitoring_baseline.json
{
  "period": "2026-03",
  "rows": 163911,
  "median_ctr": 0.0,
  "median_impressions": 108.0,
  "median_ctr_gap": 0.002926888731928216,
  "median_baseline_score": 0.007507321372320689,
  "share_low_ctr_for_position": 0.8042596287009414
}


In [17]:
# Verify that the exported files exist and can be read back.

assert queue_path.exists(), "Queue CSV was not created."
assert metrics_path.exists(), "Monitoring JSON was not created."

queue_check = pd.read_csv(queue_path)

with open(metrics_path, "r") as f:
    metrics_check = json.load(f)

print("Export verification passed.")
print("Queue rows:", len(queue_check))
print("Queue columns:", list(queue_check.columns))
print("Monitoring period:", metrics_check["period"])

Export verification passed.
Queue rows: 163911
Queue columns: ['priority_rank', 'client_hash_id', 'content_hash_id', 'gsc_avg_position', 'gsc_impressions', 'observed_ctr', 'benchmark_ctr', 'ctr_gap', 'baseline_score', 'reason_code', 'suggested_action']
Monitoring period: 2026-03


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.